<a href="https://colab.research.google.com/github/lucastiger/tfln-soliton-control/blob/main/tests/rnn_sanity_checks.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
from google.colab import userdata
import os


os.environ["GITHUB_TOKEN"] = userdata.get('GITHUB_TOKEN')

In [2]:
import os, sys

REPO_ROOT = "/content/tfln-soliton-control/"   # adjust if different

if not os.path.exists(REPO_ROOT):
    # Replace with your actual GitHub URL
    os.system(f"git clone https://$GITHUB_TOKEN@github.com/lucastiger/tfln-soliton-control.git {REPO_ROOT}")

if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)

In [ ]:
"""
ST-1: Frozen observer determinism under controller.train() (CVS-1 regression test)
Verifies that the train() override keeps the observer in eval mode, making its outputs deterministic during frozen operation.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver, PIRNNController

torch.manual_seed(0)
config = ModelConfig()
observer = PIRNNObserver(config)
controller = PIRNNController(config, observer, freeze_observer=True)

x = torch.randn(4, config.W, 1)
context = torch.randn(4, config.n_context)
delta_cmd = torch.zeros(4, 1)
target_state = torch.zeros(4, dtype=torch.long)

# Simulate the state controller.train() creates before each training epoch
controller.train()

with torch.no_grad():
    out_a = controller(x, context, delta_cmd, target_state)
    out_b = controller(x, context, delta_cmd, target_state)

h_a = out_a["logits"]
h_b = out_b["logits"]
assert torch.allclose(h_a, h_b), (
    f"Frozen observer outputs are non-deterministic under controller.train(): "
    f"max diff = {(h_a - h_b).abs().max().item():.6f}"
)
print("PASS: frozen observer is deterministic under controller.train()")

# Verify observer is in eval mode
assert not controller.observer.training, "FAIL: observer must be in eval mode when frozen"
print("PASS: observer.training == False after controller.train()")

PASS: frozen observer is deterministic under controller.train()
PASS: observer.training == False after controller.train()


In [ ]:
"""
ST-2: delta_cmd gradient path — non-zero norm, correct isolation (G3/G5)
Verifies the complete MPC gradient path and that no observer gradient leaks through when frozen.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver, PIRNNController

torch.manual_seed(42)
config = ModelConfig()
observer = PIRNNObserver(config)
controller = PIRNNController(config, observer, freeze_observer=True)
controller.eval()

x = torch.randn(2, config.W, 1)
context = torch.randn(2, config.n_context)
delta_cmd = torch.randn(2, 1, requires_grad=True)
target_state = torch.zeros(2, dtype=torch.long)

out = controller(x, context, delta_cmd, target_state)
out["act_logits"].mean().backward()

# G5: delta_cmd must receive non-zero gradient
assert delta_cmd.grad is not None, "delta_cmd.grad is None"
grad_norm = delta_cmd.grad.norm().item()
assert grad_norm > 0.0, f"delta_cmd.grad is zero (norm={grad_norm})"
print(f"PASS: delta_cmd.grad.norm = {grad_norm:.6f}")

# G3: no observer parameter should have accumulated a gradient
for name, param in controller.observer.named_parameters():
    assert param.grad is None, f"G3 violation: observer param '{name}' has grad when frozen"
print("PASS: no observer gradients accumulated (G3 satisfied)")

# G2: verify requires_grad=False on all observer params
for name, param in controller.observer.named_parameters():
    assert not param.requires_grad, f"G2 violation: '{name}' still requires grad"
print("PASS: all observer parameters frozen (G2 satisfied)")

PASS: delta_cmd.grad.norm = 0.003074
PASS: no observer gradients accumulated (G3 satisfied)
PASS: all observer parameters frozen (G2 satisfied)


In [ ]:
"""
ST-3: G4 truncated BPTT — hidden state retains gradient, input is detached
Verifies that the decoder hidden state carries gradient across all H steps while the autoregressive input feed is correctly cut.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver

torch.manual_seed(7)
config = ModelConfig()
observer = PIRNNObserver(config)

x = torch.randn(1, config.W, 1)
context = torch.randn(1, config.n_context)

out = observer(x, context)

# G4 check for detuning decoder: gradient must flow through hidden chain
grad_check = torch.autograd.grad(
    out["pred_detuning"].sum(),
    out["h_final"],
    retain_graph=True,
    allow_unused=True,
)[0]
assert grad_check is not None, "pred_detuning has no gradient w.r.t. h_final: hidden chain is broken"
assert grad_check.norm().item() > 0.0, "pred_detuning gradient w.r.t. h_final is zero"
print(f"PASS: detuning decoder hidden gradient norm = {grad_check.norm().item():.6f}")

# G4 check for P_trans decoder
grad_check_pt = torch.autograd.grad(
    out["pred_P_trans"].sum(),
    out["h_final"],
    retain_graph=True,
    allow_unused=True,
)[0]
assert grad_check_pt is not None and grad_check_pt.norm().item() > 0.0
print(f"PASS: P_trans decoder hidden gradient norm = {grad_check_pt.norm().item():.6f}")

PASS: detuning decoder hidden gradient norm = 0.390873
PASS: P_trans decoder hidden gradient norm = 0.420734


In [ ]:
"""
ST-4: h0_zeros dtype consistency under mixed-precision edge case (NTR-1 regression)
"""


import torch
from model.pi_rnn import ModelConfig, PIRNNObserver

config = ModelConfig()
observer = PIRNNObserver(config).double()  # simulate float64 model

# Input in float64 to match model
x = torch.randn(2, config.W, 1, dtype=torch.float64)
context = torch.randn(2, config.n_context, dtype=torch.float64)

try:
    out = observer(x, context)
    print(f"PASS: no dtype mismatch with float64 model and input, h_final dtype={out['h_final'].dtype}")
except RuntimeError as e:
    print(f"FAIL: dtype mismatch in h0 construction: {e}")

PASS: no dtype mismatch with float64 model and input, h_final dtype=torch.float64


In [ ]:
"""
ST-5: ModelConfig dimension derivation — no hardcoded sizes in fusion layers
Verifies that changing config dimensions updates all dependent layer sizes correctly.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver, PIRNNController

config = ModelConfig(gru_hidden=128, context_proj_dim=16, action_embed_dim=8, target_embed_dim=8)
observer = PIRNNObserver(config)
controller = PIRNNController(config, observer, freeze_observer=True)

x = torch.randn(2, config.W, 1)
context = torch.randn(2, config.n_context)
delta_cmd = torch.randn(2, 1)
target_state = torch.zeros(2, dtype=torch.long)

out_obs = observer(x, context)
assert out_obs["logits"].shape == (2, config.n_states), f"obs logits shape wrong: {out_obs['logits'].shape}"
assert out_obs["h_final"].shape == (2, 128), f"h_final shape wrong: {out_obs['h_final'].shape}"

out_ctrl = controller(x, context, delta_cmd, target_state)
assert out_ctrl["act_logits"].shape == (2, config.n_states), f"act_logits shape wrong"

print("PASS: all shapes update correctly with non-default ModelConfig dimensions")

PASS: all shapes update correctly with non-default ModelConfig dimensions


# Training dynamics validaiton

In [3]:
"""
Test Name: Gradient Magnitude Sanity on Encoder Outputs
Purpose: Verify gradients reaching h_final, ctx, phys_state from a downstream loss are finite and within a sane order-of-magnitude band — neither vanished (encoder disconnected/saturated) nor exploded (unstable init/scale bug).
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver

torch.manual_seed(0)

def test_gradient_magnitude_sanity():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, dropout=0.0)
    model = PIRNNObserver(config)
    model.eval()

    batch_size = 16
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)

    out = model(x, context)
    h_final, ctx, phys_state = out["h_final"], out["ctx"], out["phys_state"]
    h_final.retain_grad()
    ctx.retain_grad()
    phys_state.retain_grad()

    loss = out["logits"].sum() + out["pred_detuning"].sum() + out["pred_P_trans"].sum()
    loss.backward()

    for name, t in [("h_final", h_final), ("ctx", ctx), ("phys_state", phys_state)]:
        g = t.grad
        assert g is not None, f"{name} got no gradient"
        assert torch.isfinite(g).all(), f"{name} grad has NaN/Inf"
        norm = g.norm().item()
        assert norm > 1e-8, f"{name} grad norm vanished ({norm})"
        assert norm < 1e4, f"{name} grad norm exploded ({norm})"
        print(f"{name}: norm={norm:.4e} max|g|={g.abs().max().item():.4e} min|g|={g.abs().min().item():.4e}")

test_gradient_magnitude_sanity()

h_final: norm=1.5082e+00 max|g|=1.4221e-01 min|g|=6.4786e-05
ctx: norm=8.0449e-01 max|g|=1.1734e-01 min|g|=1.4803e-06
phys_state: norm=1.5919e-04 max|g|=4.8516e-05 min|g|=4.2519e-08


In [4]:
"""
Test Name: Temporal Sensitivity — Early Inputs Affect Late Horizon Predictions
Purpose: Confirm information from early timesteps of the input window propagates through the encoder GRU into h_final and reaches the last horizon-step prediction (encoder→hidden→decoder flow, temporal credit assignment).
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver

torch.manual_seed(1)

def test_temporal_sensitivity_early_inputs_affect_late_outputs():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, dropout=0.0)
    model = PIRNNObserver(config)
    model.eval()

    batch_size = 8
    x = torch.randn(batch_size, config.W, 1, requires_grad=True)
    context = torch.randn(batch_size, config.n_context)

    out = model(x, context)
    pred_detuning = out["pred_detuning"]  # (B, H)

    final_step = pred_detuning[:, -1].sum()
    grad_x, = torch.autograd.grad(final_step, x)

    assert torch.isfinite(grad_x).all(), "grad w.r.t. x has NaN/Inf"

    early = grad_x[:, : config.W // 4, :]
    late = grad_x[:, -config.W // 4 :, :]
    early_norm = early.norm().item()
    late_norm = late.norm().item()

    assert early_norm > 1e-8, (
        f"Gradient of pred_detuning[:,-1] w.r.t. earliest {config.W//4} input "
        f"timesteps is ~0 ({early_norm}); early info is not reaching the decoder."
    )
    print(f"early-window grad norm: {early_norm:.4e}, late-window grad norm: {late_norm:.4e}")

test_temporal_sensitivity_early_inputs_affect_late_outputs()

early-window grad norm: 1.6735e-06, late-window grad norm: 4.0350e-04


In [16]:
"""
Test Name: Truncated BPTT — Hidden-State Path vs. Detached Autoregressive Path
Purpose: Explicitly demonstrate the two halves of correct truncated BPTT: (a) pred_detuning[:,0] must not be an ancestor of pred_detuning[:,-1] (the inp = pred.detach() line severs the prediction→prediction path), and (b) h_final (root of decoder_h0) must remain an ancestor of pred_detuning[:,-1] with nonzero gradient (the hidden-state recurrence carries gradient across all H steps).
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver

torch.manual_seed(2)

def test_truncated_bptt_hidden_vs_autoregressive_path():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, dropout=0.0)
    model = PIRNNObserver(config)
    model.eval()

    batch_size = 8
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)

    out = model(x, context)
    pred_detuning = out["pred_detuning"]
    pred_p_trans = out["pred_P_trans"]
    h_final = out["h_final"]

    # NOTE: pred_detuning[:, 0:1] (a fresh slice of the returned tensor) is NOT
    # a backward-reachable ancestor of pred_detuning[:, -1].sum() -- it is a
    # sibling branch off the same CatBackward node. torch.autograd.grad with
    # allow_unused=True returns None for such siblings UNCONDITIONALLY, so
    # querying gradient "from pred_detuning[:,-1] to pred_detuning[:,0:1]"
    # cannot detect whether `.detach()` is present. That check is removed.
    #
    # What we CAN validly verify with returned tensors:
    #   (1) h_final remains an ancestor of pred_detuning[:,-1] (H-step recurrence
    #       carries gradient through the hidden-state chain), and
    #   (2) h_final is ALSO an ancestor of pred_detuning[:,0] (the single-step
    #       decoder output is grounded in the encoder, not a free constant).
    # Verification that `inp = pred.detach()` is structurally present must be
    # done by code inspection of forward(), not by a black-box gradient probe.

    grad_h_last, = torch.autograd.grad(
        pred_detuning[:, -1].sum(), h_final, retain_graph=True
    )
    assert torch.isfinite(grad_h_last).all()
    norm_last = grad_h_last.norm().item()
    assert norm_last > 1e-10, (
        f"h_final grad via H={config.H}-step recurrence vanished ({norm_last})"
    )

    grad_h_first, = torch.autograd.grad(
        pred_detuning[:, 0].sum(), h_final, retain_graph=True
    )
    assert torch.isfinite(grad_h_first).all()
    norm_first = grad_h_first.norm().item()
    assert norm_first > 1e-10, (
        f"h_final grad via 1-step decoder vanished ({norm_first}); "
        f"pred_detuning[:,0] may be ignoring the encoder."
    )

    grad_h_ptrans, = torch.autograd.grad(
        pred_p_trans[:, -1].sum(), h_final, retain_graph=True
    )
    assert torch.isfinite(grad_h_ptrans).all()
    assert grad_h_ptrans.norm().item() > 1e-10, "P_trans decoder disconnected from h_final"

    print(f"h_final grad norm (detuning step 0):  {norm_first:.4e}")
    print(f"h_final grad norm (detuning step H-1): {norm_last:.4e}")
    print(f"h_final grad norm (P_trans step H-1):  {grad_h_ptrans.norm().item():.4e}")

test_truncated_bptt_hidden_vs_autoregressive_path()

h_final grad norm (detuning step 0):  4.1500e-01
h_final grad norm (detuning step H-1): 7.1067e-03
h_final grad norm (P_trans step H-1):  1.3646e-02


In [6]:
"""
Test Name: Step-wise Gradient Diversity (Degenerate Recurrence Detection)
Purpose: Detect a GRU decoder that isn't actually evolving state step-to-step (e.g., effectively producing the same hidden state regardless of step index). Checks (a) horizon outputs vary across steps, and (b) gradients of distinct horizon-step predictions w.r.t. h_final are distinguishable.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver

torch.manual_seed(3)

def test_stepwise_gradient_diversity():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, dropout=0.0)
    model = PIRNNObserver(config)
    model.eval()

    batch_size = 8
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)

    out = model(x, context)
    pred_detuning = out["pred_detuning"]  # (B, H)
    h_final = out["h_final"]

    assert torch.isfinite(pred_detuning).all()

    # (a) outputs must vary across the H horizon steps
    step_means = pred_detuning.mean(dim=0)  # (H,)
    spread = step_means.std().item()
    assert spread > 1e-6, (
        f"pred_detuning is ~constant across H={config.H} steps (std={spread:.3e}); "
        f"decoder recurrence may be degenerate (hidden state not evolving)."
    )

    # (b) gradients of distinct steps w.r.t. h_final must differ
    steps = [0, config.H // 2, config.H - 1]
    grads = []
    for s in steps:
        g, = torch.autograd.grad(pred_detuning[:, s].sum(), h_final, retain_graph=True)
        grads.append(g)

    for i in range(len(grads)):
        for j in range(i + 1, len(grads)):
            diff = (grads[i] - grads[j]).norm().item()
            denom = grads[i].norm().item() + grads[j].norm().item() + 1e-12
            rel = diff / denom
            assert rel > 1e-4, (
                f"d(pred_detuning[:,{steps[i]}])/d(h_final) ≈ "
                f"d(pred_detuning[:,{steps[j]}])/d(h_final) (rel diff={rel:.3e}); "
                f"step-distinct sensitivity missing (degenerate recurrence)."
            )
            print(f"steps {steps[i]} vs {steps[j]}: rel grad diff = {rel:.4e}")

test_stepwise_gradient_diversity()

steps 0 vs 4: rel grad diff = 8.3901e-01
steps 0 vs 7: rel grad diff = 9.5891e-01
steps 4 vs 7: rel grad diff = 5.9624e-01


In [7]:
"""
Test Name: Controller Gradient Usability — delta_cmd
Purpose: Validate the core MPC gradient path: d(P(single_soliton at horizon)) / d(delta_cmd) must be finite, nonzero, and bounded, since this is the signal used for gradient-ascent MPC solves.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver, PIRNNController, SINGLE_SOLITON_IDX

torch.manual_seed(4)

def test_controller_delta_cmd_gradient_usability():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, action_embed_dim=8, target_embed_dim=4,
                          dropout=0.0)
    observer = PIRNNObserver(config)
    controller = PIRNNController(config, observer, freeze_observer=True)
    controller.eval()

    batch_size = 16
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)
    delta_cmd = torch.randn(batch_size, 1, requires_grad=True)
    target_state = torch.full((batch_size,), SINGLE_SOLITON_IDX, dtype=torch.long)

    out = controller(x, context, delta_cmd=delta_cmd, target_state=target_state)
    objective = torch.log_softmax(out["act_logits"], dim=-1)[:, SINGLE_SOLITON_IDX].mean()

    grad, = torch.autograd.grad(objective, delta_cmd)

    assert torch.isfinite(grad).all(), "delta_cmd grad has NaN/Inf"
    norm = grad.norm().item()
    max_abs = grad.abs().max().item()
    assert norm > 1e-8, f"delta_cmd grad vanished (norm={norm})"
    assert norm < 1e4 and max_abs < 1e4, f"delta_cmd grad exploded (norm={norm}, max={max_abs})"
    print(f"delta_cmd grad norm={norm:.4e}, max|g|={max_abs:.4e}")

test_controller_delta_cmd_gradient_usability()

delta_cmd grad norm=2.1448e-03, max|g|=1.1202e-03


In [10]:
"""
Test Name: Controller↔Observer Interface Gradient (Frozen vs. Unfrozen)
Purpose: Validate the freeze/unfreeze contract at the controller-observer boundary: with freeze_observer=True, observer parameters receive no gradient while delta_cmd still does; with freeze_observer=False, gradient does reach observer parameters through the shared h_final/ctx path.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver, PIRNNController

torch.manual_seed(5)

def test_controller_observer_interface_gradient():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, action_embed_dim=8, target_embed_dim=4,
                          dropout=0.0)

    batch_size = 8
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)
    target_state = torch.randint(0, config.n_states, (batch_size,))

    # --- frozen observer ---
    delta_cmd_f = torch.randn(batch_size, 1, requires_grad=True)
    obs_f = PIRNNObserver(config)
    ctrl_f = PIRNNController(config, obs_f, freeze_observer=True)
    ctrl_f.eval()

    out_f = ctrl_f(x, context, delta_cmd=delta_cmd_f, target_state=target_state)
    (out_f["act_logits"].sum() + out_f["act_pred_detuning"].sum()).backward()

    for p in obs_f.parameters():
        assert p.grad is None, "Observer param got gradient despite freeze_observer=True"
    assert delta_cmd_f.grad is not None and torch.isfinite(delta_cmd_f.grad).all()
    assert delta_cmd_f.grad.norm().item() > 1e-8, "delta_cmd grad vanished under frozen observer"

    # --- unfrozen observer ---
    delta_cmd_u = torch.randn(batch_size, 1, requires_grad=True)
    obs_u = PIRNNObserver(config)
    ctrl_u = PIRNNController(config, obs_u, freeze_observer=False)
    ctrl_u.eval()

    out_u = ctrl_u(x, context, delta_cmd=delta_cmd_u, target_state=target_state)
    (out_u["act_logits"].sum() + out_u["act_pred_detuning"].sum()).backward()

    grad_norms = [p.grad.norm().item() for p in obs_u.parameters() if p.grad is not None]
    assert len(grad_norms) > 0, "No observer params received gradient with freeze_observer=False"
    assert any(g > 1e-8 for g in grad_norms), "All observer grads ~0 with freeze_observer=False"
    assert all(torch.isfinite(p.grad).all() for p in obs_u.parameters() if p.grad is not None)

    print(f"frozen: delta_cmd grad norm={delta_cmd_f.grad.norm().item():.4e}")
    print(f"unfrozen: max observer param grad norm={max(grad_norms):.4e} "
          f"({len(grad_norms)} params with grad)")

test_controller_observer_interface_gradient()

frozen: delta_cmd grad norm=7.5833e-02
unfrozen: max observer param grad norm=3.8186e+00 (20 params with grad)


In [11]:
"""
Test Name: Minimal Training Loop — Loss Decreases
Purpose: End-to-end trainability sanity: optimize a combined classification + forecasting loss on a fixed batch and confirm it decreases, ruling out dead gradients, sign errors, or pathological loss scaling that wouldn't show up in single-step gradient checks.
"""

import torch
from torch import nn
from model.pi_rnn import ModelConfig, PIRNNObserver

torch.manual_seed(6)

def test_minimal_training_loop_loss_decreases():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, dropout=0.0)
    model = PIRNNObserver(config)
    model.train()

    batch_size = 16
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)
    target_labels = torch.randint(0, config.n_states, (batch_size,))
    target_detuning = torch.randn(batch_size, config.H)
    target_p_trans = torch.randn(batch_size, config.H)

    ce, mse = nn.CrossEntropyLoss(), nn.MSELoss()
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)

    losses = []
    n_steps = 60
    for step in range(n_steps):
        opt.zero_grad()
        out = model(x, context)
        loss = (ce(out["logits"], target_labels)
                + mse(out["pred_detuning"], target_detuning)
                + mse(out["pred_P_trans"], target_p_trans))
        loss.backward()
        opt.step()
        losses.append(loss.item())
        if step % 10 == 0 or step == n_steps - 1:
            print(f"step {step:3d}: loss={loss.item():.6f}")

    assert all(l == l and abs(l) < 1e6 for l in losses), "Loss became NaN/Inf during training"
    early, late = sum(losses[:5]) / 5, sum(losses[-5:]) / 5
    assert late < early, f"Loss did not decrease (early={early:.6f}, late={late:.6f})"
    print(f"early avg={early:.6f}, late avg={late:.6f}")

test_minimal_training_loop_loss_decreases()

step   0: loss=3.806249
step  10: loss=3.722489
step  20: loss=3.600379
step  30: loss=3.400945
step  40: loss=3.109915
step  50: loss=2.665600
step  59: loss=2.346342
early avg=3.788625, late avg=2.411399


In [12]:
"""
Test Name: Output Distribution Sanity (Observer + Controller)
Purpose: Catch silent collapse (constant outputs) or explosion (runaway magnitudes) across all major returned tensors on a realistic batch size.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver, PIRNNController

torch.manual_seed(7)

def _check(name, t, min_std=1e-6, max_abs=1e4):
    assert torch.isfinite(t).all(), f"{name} has NaN/Inf"
    std, mean, mx = t.float().std().item(), t.float().mean().item(), t.float().abs().max().item()
    assert mx < max_abs, f"{name} exploded: max|.|={mx:.3e}"
    assert std > min_std, f"{name} collapsed: std={std:.3e}"
    print(f"{name}: mean={mean:.3e} std={std:.3e} max|.|={mx:.3e}")

def test_output_distribution_sanity():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, action_embed_dim=8, target_embed_dim=4,
                          dropout=0.0)
    observer = PIRNNObserver(config)
    controller = PIRNNController(config, observer, freeze_observer=True)
    observer.eval()
    controller.eval()

    batch_size = 64
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)

    out_obs = observer(x, context)
    for key in ["logits", "pred_detuning", "pred_P_trans", "phys_state", "h_final", "ctx"]:
        _check(f"observer.{key}", out_obs[key])

    delta_cmd = torch.randn(batch_size, 1)
    target_state = torch.randint(0, config.n_states, (batch_size,))
    out_ctrl = controller(x, context, delta_cmd=delta_cmd, target_state=target_state)
    for key in ["act_pred_detuning", "act_logits"]:
        _check(f"controller.{key}", out_ctrl[key])

test_output_distribution_sanity()

observer.logits: mean=5.990e-02 std=3.955e-02 max|.|=1.304e-01
observer.pred_detuning: mean=-3.017e-02 std=1.666e-02 max|.|=1.150e-01
observer.pred_P_trans: mean=-3.508e-01 std=2.824e-02 max|.|=3.707e-01
observer.phys_state: mean=-7.527e-02 std=2.026e-01 max|.|=8.985e-01
observer.h_final: mean=-4.133e-02 std=1.502e-01 max|.|=4.602e-01
observer.ctx: mean=3.176e-02 std=4.613e-01 max|.|=1.168e+00
controller.act_pred_detuning: mean=1.318e-02 std=1.538e-02 max|.|=9.486e-02
controller.act_logits: mean=3.407e-02 std=6.789e-02 max|.|=2.270e-01


In [13]:
"""
Test Name: Long-Horizon Numerical Stability Stress Test
Purpose: Stress the autoregressive decoder over a longer horizon (H=64) and larger-magnitude inputs to detect saturation/collapse or gradient blow-up that only manifests over extended rollout — not visible at H=8.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver

torch.manual_seed(8)

def test_long_horizon_numerical_stability():
    config = ModelConfig(W=20, H=64, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, dropout=0.0)
    model = PIRNNObserver(config)
    model.eval()

    batch_size = 8
    x = torch.randn(batch_size, config.W, 1) * 10.0
    context = torch.randn(batch_size, config.n_context) * 5.0

    out = model(x, context)
    pred_detuning, pred_p_trans, h_final = out["pred_detuning"], out["pred_P_trans"], out["h_final"]

    assert torch.isfinite(pred_detuning).all(), "pred_detuning NaN/Inf under stress input"
    assert torch.isfinite(pred_p_trans).all(), "pred_P_trans NaN/Inf under stress input"

    late_std = pred_detuning[:, config.H // 2:].std().item()
    assert late_std > 1e-8, (
        f"pred_detuning collapses in late horizon (std={late_std:.3e} over "
        f"steps {config.H//2}..{config.H}); possible hidden-state saturation."
    )

    grad_h, = torch.autograd.grad(pred_detuning[:, -1].sum(), h_final, retain_graph=True)
    assert torch.isfinite(grad_h).all(), "Gradient through long-horizon decoder has NaN/Inf"
    grad_norm = grad_h.norm().item()
    assert grad_norm < 1e6, f"Gradient exploded over long horizon (norm={grad_norm})"
    print(f"H={config.H}: late-horizon std={late_std:.4e}, final-step grad norm={grad_norm:.4e}")

test_long_horizon_numerical_stability()

H=64: late-horizon std=3.1617e-08, final-step grad norm=1.2402e-13


In [14]:
"""
Test Name: MPC Gradient-Ascent Convergence on delta_cmd
Purpose: Go beyond "gradient exists" to "gradient is usable for the actual MPC solve": iterative normalized gradient ascent on delta_cmd should monotonically-ish increase the predicted probability of the target (single-soliton) state, confirming the action→objective landscape is navigable.
"""

import torch
from model.pi_rnn import ModelConfig, PIRNNObserver, PIRNNController, SINGLE_SOLITON_IDX

torch.manual_seed(9)

def test_mpc_gradient_ascent_increases_target_probability():
    config = ModelConfig(W=20, H=8, n_context=4, n_states=7, context_proj_dim=8,
                          phys_state_dim=4, gru_hidden=32, gru_layers=2,
                          decoder_hidden=16, action_embed_dim=8, target_embed_dim=4,
                          dropout=0.0)
    observer = PIRNNObserver(config)
    controller = PIRNNController(config, observer, freeze_observer=True)
    controller.eval()

    batch_size = 8
    x = torch.randn(batch_size, config.W, 1)
    context = torch.randn(batch_size, config.n_context)
    target_state = torch.full((batch_size,), SINGLE_SOLITON_IDX, dtype=torch.long)

    def objective(delta):
        out = controller(x, context, delta_cmd=delta, target_state=target_state)
        return torch.softmax(out["act_logits"], dim=-1)[:, SINGLE_SOLITON_IDX].mean()

    delta_cmd = torch.zeros(batch_size, 1, requires_grad=True)
    lr, n_steps = 0.05, 30
    obj_values = []

    for step in range(n_steps):
        obj = objective(delta_cmd)
        obj_values.append(obj.item())
        grad, = torch.autograd.grad(obj, delta_cmd)
        grad_dir = grad / (grad.norm() + 1e-8)
        with torch.no_grad():
            new_delta = delta_cmd + lr * grad_dir
        delta_cmd = new_delta.detach().requires_grad_(True)
        if step % 5 == 0:
            print(f"step {step:2d}: P(single_soliton)={obj.item():.4f}")

    assert all(v == v for v in obj_values), "Objective became NaN during ascent"
    assert obj_values[-1] > obj_values[0], (
        f"Normalized gradient ascent on delta_cmd did not increase P(single_soliton) "
        f"(start={obj_values[0]:.4f}, end={obj_values[-1]:.4f}); MPC objective "
        f"landscape is flat or gradient sign/scale is unusable."
    )
    print(f"P(single_soliton): {obj_values[0]:.4f} -> {obj_values[-1]:.4f}")

test_mpc_gradient_ascent_increases_target_probability()

step  0: P(single_soliton)=0.1423
step  5: P(single_soliton)=0.1425
step 10: P(single_soliton)=0.1428
step 15: P(single_soliton)=0.1431
step 20: P(single_soliton)=0.1433
step 25: P(single_soliton)=0.1436
P(single_soliton): 0.1423 -> 0.1437
